In [0]:
df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/default/my_files/customers-1000.csv")

display(df)

In [0]:
df = df.toDF(*[c.replace(" ", "_").lower() for c in df.columns])

display(df)

In [0]:
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.customers_delta_demo")

In [0]:
# Read Delta Table
customers_df = spark.table("workspace.default.customers_delta_demo")

display(customers_df)

In [0]:
# ACID Transaction (Update Data)

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
spark,
"workspace.default.customers_delta_demo"
)

delta_table.update(
condition = "customer_id = 'ED014d010c7ab0c'",
set = {"city": "'Chennai'"}
)

In [0]:
# View Delta Log History

delta_table.history().show()

In [0]:
# View Delta Log History

%sql
DESCRIBE HISTORY workspace.default.customers_delta_demo


In [0]:
# Time Travel (Read Older Version)
spark.read.format("delta") \
.option("versionAsOf",0) \
.table("workspace.default.customers_delta_demo") \
.display()

In [0]:
# Schema Enforcement Example

# Create data with a new column not in the table.

data_new = [("999999","Test","User","India",25)]

columns_new = [
"customer_id",
"first_name",
"last_name",
"country",
"age"
]

df_new = spark.createDataFrame(data_new,columns_new)

df_new.write.format("delta") \
.mode("append") \
.saveAsTable("workspace.default.customers_delta_demo")

In [0]:
# Schema Evolution (Fix the Error)
df_new.write.format("delta") \
.option("mergeSchema","true") \
.mode("append") \
.saveAsTable("workspace.default.customers_delta_demo")

In [0]:
spark.table("workspace.default.customers_delta_demo").display()